# Demonstração do Pipeline Estatístico do Projeto

Este notebook é uma demonstração **simples e direta** da arquitetura do nosso projeto, alinhada com as três fases principais estabelecidas nos diagramas metodológicos:

1. **Etapa 1: Validação Humana** (Métricas de concordância)
2. **Calibragem** (Ancoragem empírica da ROPE)
3. **Etapa 2: Comparação de Modelos (LLMs)** (Inferência Bayesiana, Forest Plot e Heatmap)

Utilizaremos **dados fictícios** simulando a mesma escala Likert (1 a 4) e a mesma estrutura pareada adotada na dissertação.

> **Nota**: Os cálculos bayesianos utilizam o nosso pacote customizado `util_est_bayesiana.py`, que atua como um wrapper sobre a lógica do `baycomp` mas permite gráficos e limiares específicos para a nossa análise.

In [ ]:
# ── Importações e Dependências ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

# Adiciona o diretório fonte (src) para importar nosso pacote customizado
sys.path.append("../src")
from util_est_bayesiana import (Comparacao, grafico_diferencas, heatmap, matriz_pares)

def discretizar(x):
    return np.clip(np.round(x) + 2, 1, 4).astype(int)

def kappa_w(x, y):
    # Simulação simples de concordância para o demo
    return np.mean(x == y)

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)

print("Ferramentas carregadas com sucesso!")

---
## Etapa 1: Validação (Concordância Interavaliadores)

O projeto avalia extrações através de juízes. Primeiramente, medimos o quão consistente é a equipe de **especialistas humanos** usando o $\kappa_w$ (Cohen's Kappa com pesos ordinais ou métrica similar de concordância).

Isso estabelece um "piso de ruído": divergências normais que ocorrem até mesmo entre os humanos mais treinados.

In [ ]:
# Simulação de 3 especialistas (E1, E2, E3) avaliando 300 documentos
SEED = 42
N_DOCS = 300
rng = np.random.default_rng(SEED)

# Dificuldade real do documento (latente)
dificuldade = rng.normal(0, 1.0, N_DOCS)

# Notas dos especialistas (1 a 4) com pequenos vieses de severidade e ruído
H_E1 = discretizar(dificuldade - 0.10 + rng.normal(0, 0.40, N_DOCS))
H_E2 = discretizar(dificuldade + 0.05 + rng.normal(0, 0.40, N_DOCS))
H_E3 = discretizar(dificuldade + 0.00 + rng.normal(0, 0.45, N_DOCS))

# Extraindo a métrica de concordância kappa_w (implementada em nosso pacote)
kw_12 = kappa_w(H_E1, H_E2)
kw_13 = kappa_w(H_E1, H_E3)
kw_23 = kappa_w(H_E2, H_E3)

print("Concordância entre Avaliadores Humanos (Kappa kw):")
print(f"  E1 vs E2: {kw_12:.3f}")
print(f"  E1 vs E3: {kw_13:.3f}")
print(f"  E2 vs E3: {kw_23:.3f}")
print(f"  -> Média: {np.mean([kw_12, kw_13, kw_23]):.3f} (Sinal verde para Validação)")

---
## Calibragem: Definição Empírica da ROPE

Na estatística bayesiana, a ROPE (*Region of Practical Equivalence*) nunca deve ser um chute. Nós ancoramos esse valor na **divergência humana real**. Se os próprios avaliadores especialistas discordam em média $0.15$ pontos na escala, então diferenças menores que essa entre dois protocolos não são significativas, são apenas ruído.

In [ ]:
# Vamos calcular a divergência absoluta média entre os pares humanos
dif_12 = np.abs(H_E1 - H_E2).mean()
dif_13 = np.abs(H_E1 - H_E3).mean()
dif_23 = np.abs(H_E2 - H_E3).mean()

divergencia_media = np.mean([dif_12, dif_13, dif_23])

# Arredondamos levemente para cima para estabelecer a margem de equivalência (ROPE)
ROPE_CALIBRADA = float(np.ceil(divergencia_media * 100) / 100)

print(f"Divergências médias: E1-E2={dif_12:.3f}, E1-E3={dif_13:.3f}, E2-E3={dif_23:.3f}")
print(f"Divergência global = {divergencia_media:.4f}")
print(f"-> ROPE Calibrada e fixada empiricamente em: {ROPE_CALIBRADA}")

---
## Etapa 2: Comparação de Modelos (Protocolos de Extração)

Agora comparamos os protocolos reais (ex: GPT-4, Claude-3, Llama-3). A análise injeta a `ROPE_CALIBRADA` e as notas na nossa classe `ComparacaoPareada`, para obtermos as probabilidades e o IC $95\%$.

In [ ]:
# Geração de dados de modelos simulando uma diferença no desempenho latente
modelos = {
    "GPT-4": 0.60,       # Fortíssimo
    "Claude-3": 0.55,    # Quase equivalente ao GPT-4
    "Llama-3": 0.15,     # Médio
    "Base": -0.20        # Baseline mais fraco
}

notas_modelos = pd.DataFrame({
    nome: discretizar(dificuldade + efeito + rng.normal(0, 0.35, N_DOCS))
    for nome, efeito in modelos.items()
})

# Executando uma comparação específica: GPT-4 vs Base
cmp_exemplo = Comparacao(notas_modelos["GPT-4"], notas_modelos["Base"], rope=ROPE_CALIBRADA)
print("Exemplo de Comparação Direta (GPT-4 vs Base):")
print(f"  Delta Médio: {cmp_exemplo.diferenca_media:+.3f}")
print(f"  P(Equivalência) com ROPE {ROPE_CALIBRADA}: {cmp_exemplo.probabilidades['p_rope']:.4f}")
print(f"  P(Superioridade): {cmp_exemplo.probabilidades['p_esquerda']:.4f}")

### Visualização 1: Forest Plot Bayesiano
Demonstra os intervalos $95\%$ para as diferenças, pintando barras de Verde/Vermelho/Azul se o modelo for classificado como Superior/Inferior/Equivalente, respeitando o Limiar de `0.95`.

In [ ]:
# Para desenhar o Forest Plot e o Heatmap, calculamos todas as relações
# (Note que matriz_pares já encapsula a Comparacao para todas as permutações)
matriz_completa = matriz_pares(
    notas_modelos, 
    nomes=list(modelos.keys()), 
    rope=ROPE_CALIBRADA, 
    limiar=0.95
)

# Vamos filtrar alguns pares para o Forest Plot (não direcional, pegando apenas os triangulares)
# Queremos comparar todos contra o Baseline (Base)
pares_interesse = matriz_completa[matriz_completa["coluna"] == "Base"].copy()

figura_fp, _ = grafico_diferencas(
    pares_interesse, 
    titulo=f"Forest Plot: Modelos vs Baseline (ROPE={ROPE_CALIBRADA})"
)
plt.show()

### Visualização 2: Heatmap Categórico-Numérico
Mostra de uma só vez a matriz de quem vence de quem, usando a **cor** para a categoria (Verde=Superior, etc), e o **alfa** e **número** para a força da probabilidade posterior.

In [ ]:
# O heatmap exibe a mesma informação mas numa visão global de contrastes 4-Way
figura_hm, _ = heatmap(
    matriz_completa,
    titulo="Heatmap de Contrastes (Todos contra Todos)",
    rotulo="Modelo Avaliado"
)
plt.show()

print("\n--- Conclusão da Demonstração ---")
print("Toda a base estatística está amarrada e visualmente clara para apresentação.")